In [17]:
import math
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.patches import Ellipse
from itertools import chain
import time
from tqdm import tqdm
from operator import attrgetter
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from scipy.stats import chi2
from scipy.spatial.distance import pdist

NOTE: merge function not the same as previous one!

In [58]:
PMIN = 0.75
PMAX = 6

In [57]:
# surface functions, merge function, and other helper functions

def surface_single(x, U, A, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for a single point.

    Parameters
    ----------
    x : np.ndarray, shape (d,)
        The query point.
    U : np.ndarray, shape (d, d)
        Eigenvectors (columns) defining the ellipsoid orientation.
    A : np.ndarray, shape (d,)
        Semi-axis lengths along each eigenvector direction.
    p : float or np.ndarray, shape (d)
        power/exponent of the ellipsoid axes.
    k : float
        confidence value

    Returns
    -------
    float
        < 0 if x is inside the ellipsoid
        = 0 if x is on the surface
        > 0 if x is outside the ellipsoid
    """
    x_local = U.T @ x        # Project into ellipsoid's local frame, shape (d,)
    return np.sum(np.abs(x_local / A) ** p) - (k ** 2)

def surface_multi(xs, Us, As, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for n points,
    each against its own ellipsoid.

    Parameters
    ----------
    xs : np.ndarray, shape (n, d)
        Query points.
    Us : np.ndarray, shape (n, d, d)
        Per-ellipsoid eigenvector matrices.
    As : np.ndarray, shape (n, d)
        Per-ellipsoid semi-axis lengths.

    Returns
    -------
    np.ndarray, shape (n,)
        Surface function value for each (point, ellipsoid) pair.
        < 0 inside, = 0 on surface, > 0 outside.
    """
    xs_local = np.einsum('ndi, ni -> nd', Us.swapaxes(1,2), xs)  # (n, d)
    return np.sum(np.abs(xs_local / As) ** p, axis=1) - (k ** 2)              # (n,)

def merge_nodes(x, y): # NOTE: New P is currently avg of the 2 that is (x.p + y.p) / 2
    new_n = x.n + y.n
    dist = x.M - y.M
    new_M = ((x.n * x.M) + (y.n * y.M)) / (new_n)
    new_S = ((x.n / new_n) * x.S) + ((y.n / new_n) * y.S) + (((x.n * y.n)/ (new_n ** 2)) * (np.outer(dist, dist)))
    eigen_value, eigen_vector = np.linalg.eigh(new_S)
    # reverse the order of the eigenvalue/vectors to be in DESCENDING order
    eigen_value = eigen_value[::-1]
    eigen_vector = eigen_vector[:, ::-1]
    # calculate new width that based on confidence ellipsoid
    confidence = new_n / (new_n + x.dim)
    chi = chi2.ppf(confidence, x.dim)
    new_A = np.sqrt(np.abs(eigen_value) * chi)
    eigen_value[eigen_value == 0] = x.eps
    avg_eig = np.mean(eigen_value)
    new_p = 2 * np.sqrt(np.abs(avg_eig / eigen_value))
    new_p = np.clip(new_p, a_min=PMIN, a_max=PMAX)
    new_A[new_A == 0] = x.eps

    return Node(dimension=x.dim, label=x.label, A=new_A, n=new_n, M=new_M, S=new_S, U=eigen_vector, eps=x.eps, alpha=x.alpha, p=new_p)

def print_stat(arr, name):
    print(f"{name} Mean:", np.mean(arr))
    print(f"{name} Max:", np.max(arr))
    print(f"{name} Min:", np.min(arr))

def print_time(arr, name):
    print(f"action: {name}")
    print(f"count: {len(arr)}")
    print(f"total: {sum(arr)}")
    print(f"avg: {sum(arr)/len(arr)}")

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]



In [56]:
class Node():
    def __init__(self,
        dimension, label,
        A, p, eps, alpha,
        n=1,
        M=None,
        S=None,
        U=None):
        """
        A node contain information about a single ellipsoid such as its centre, cov matrix, and the axis length andd direction.
        It can be "updated" with a point to move it to learn and cover more data point.
        
        Parameters
        A (np.ndarray) (dim) : The length of each axis. Not necessary sorted.  np array 
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function.
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        n (int) : The number of data points this node has learnt from.
        U (np.ndarray) (dim, dim) : The eigenvectors of the cov matrix. NOT transposed that is U[i] contain the ith eigenvector.
        M (np.ndarray) (dim) : The mean/center of the ellipsoid
        S (np.ndarray) (dim, dim) : The cov matrix of the ellipsoid. 
        """

        self.dim = dimension
        self.label = label
        self.A = A
        self.p = p
        self.eps = eps
        self.alpha = alpha
        self.n = n
        self.confidence = n / (n + self.dim)
        self.chi = chi2.ppf(self.confidence, self.dim)

        self.U = np.eye(dimension) if U is None else U
        self.M = np.array([0.] * dimension) if M is None else M
        self.S = np.zeros((dimension, dimension)) if S is None else S

    def update(self, x, parent):
        """
        Update the node with a new data point x.
        Performing the following steps.
        1. Calculate the new mean and cov matrix of the node including the new point
        2. Calculate the eigenvalue/vector of the new cov matrix
        3. Update the width of the axes of the ellipsoid
        4. Calculate the growth criterion to check whether the updated ellipsoid cover the new data point
        5. If the growth criterion is acceptable the node is allowed to grow/update to include this new data point,
           else the node is not updated and a new node is added to cover the new datapoint instead.

        Parameter
        x (np.ndarray) (dim) : A new data point the model need to learn/update from.
        parent (VEBF) : The VEBF model this node belong to. We only need this in case we need to add a new node to the VEBF model. 
        """

        # Calculate new mean and cov matrix
        n = self.n
        M = self.M
        alpha = n / (n + 1)
        beta = x / (n + 1)
        M_new = (alpha * M) + beta
        k_1 = (np.outer(x, x)/(n + 1)) - np.outer(M_new, M_new) + np.outer(M, M)
        k_2 = - (np.outer(M, M) / (n + 1))
        k = k_1 + k_2
        S_new = (alpha * self.S) + k

        eigen_value, eigen_vector = np.linalg.eigh(S_new)
        # reverse the order of the eigenvalue/vectors to be in DESCENDING order
        eigen_value = eigen_value[::-1]
        eigen_vector = eigen_vector[:, ::-1]

        # calculate new width with adaptively by combining fixed and dynamic width update rules
        """beta = 1 - self.alpha
        fixed = np.sqrt(np.pi * 2 * np.abs(eigen_value))
        dynamic = self.A + ((M_new - self.M) @ eigen_vector.T)
        new_A = (self.alpha * fixed) + (beta * dynamic)
        new_A[new_A == 0] = self.eps"""

        # calculate new width that based on confidence ellipsoid
        new_A = np.sqrt(np.abs(eigen_value) * self.chi)
        new_A[new_A == 0] = self.eps

        # clculate updated exponent of each axis
        eigen_value[eigen_value == 0] = self.eps
        avg_eig = np.mean(eigen_value)
        new_p = 2 * np.sqrt(np.abs(avg_eig / eigen_value))
        new_p = np.clip(new_p, a_min=PMIN, a_max=PMAX)
        
        gc = surface_single(x - M_new, eigen_vector, self.A, self.p)
        if gc <= 0:
            # update the current node
            self.M = M_new
            self.U = eigen_vector
            self.S = S_new
            self.A = new_A
            self.n += 1
            self.p = new_p
            return self
        else:
            # add new node
            new_node = Node(self.dim, self.label, M=x, eps=self.eps, A=parent.default_width, alpha=self.alpha, p=np.array([2.] * self.dim))
            parent.nodes[self.label].append(new_node)
            return new_node

    def __str__(self):
        return f"A node covering {self.n} points."

In [12]:
class VEBF():
    def __init__(self, dimension, merge_parameter=0, eps=0.00001, default_width=None, alpha=0.55, p=None):
        """
        The Versatile Ellipsoid Basis Function model.
        Geometrically, the model learn by covering the datapoints with ellipsoids to learn the distribution of the data.
        
        Parameter
        dimension (int) : Number of dimension of input feature.
        merge_parameter (float) : The hyperparameter that controll when two ellipsoids/nodes will be merged.
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function. (dim) np array
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        default_width (np.ndarray) (dim) : The default axis width of the ellipsoid.
        """
        self.dim = dimension
        self.merge_parameter = merge_parameter
        self.nodes = {}
        self.eps = eps
        self.alpha = alpha
        self.default_width = np.array([1.] * dimension) if default_width is None else default_width
        self.p = np.array([2.] * dimension) if p is None else p # default to "normal" hyperellipsoid with p=2

    def find_shortest(self, x, y):
        # TODO: vectorize this
        min_dist = float('inf')
        min_node = None
        for node in self.nodes[y]:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    # NOTE: can combine this with find_shortest by using y=None as default value
    def find_shortest_all(self, x, nodes=None):
        min_dist = float('inf')
        min_node = None
        nodes = self.get_nodes() if nodes is None else nodes
        for node in nodes:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    def check_merge(self, x):
        """
        Check for pair of nodes which can be merged.
        Only consider the "latest" node, x

        Parameters:
        x (Node): The node which has just been updated or created.
        """
        nodes = self.nodes[x.label]
        if len(nodes) == 1:
            return
        nodes.remove(x)

        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])
        
        scores = surface_multi(x.M - Ms, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(x, y)
            nodes.append(new_node)
            self.check_merge(new_node)
            return

        n = len(nodes)
        U = x.U
        A = x.A
        P = x.p
        Us = np.broadcast_to(U, (n,) + U.shape)
        As = np.broadcast_to(A, (n,) + A.shape)
        Ps = np.broadcast_to(P, (n,) + P.shape)

        scores = surface_multi(Ms - x.M, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(y, x)
            nodes.append(new_node)
            self.check_merge(new_node)
            return
            
        nodes.append(x)

    def get_nodes(self):
        return list(chain.from_iterable(self.nodes.values()))

    def predict(self, x):

        nodes = self.get_nodes()
        assert len(nodes) > 0, "Can't predict with empty VEBF."
        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])

        scores = surface_multi(x - Ms, Us, As, Ps)
        idx = np.argsort(scores)
        result = nodes[idx[0]].label
        return result
        """ OLD prediction by euclidean

        candidates = [node for node, score in zip(nodes, scores) if score <= 0]

        if len(candidates) == 0:
            closest = self.find_shortest_all(x)
            return closest.label
        return self.find_shortest_all(x, candidates).label"""
            
    def train(self, x, y):
        if y not in self.nodes: # Add a new node for unseen label.
            self.nodes[y] = [Node(dimension=self.dim, label=y, M=x, eps=self.eps, A=self.default_width, alpha=self.alpha, p=self.p)]
        else: # Update closest existing node for existing label.
            shortest = self.find_shortest(x, y)
            latest = shortest.update(x, self) # latest is the node with latest change, which need to checked for merge
            self.check_merge(latest)

    def __str__(self):
        total = sum(len(x) for x in self.nodes.values())
        string = f"A vebf with {total} nodes.\n"
        for label, nodes in self.nodes.items():
            for node in nodes:
                string += f"Label {label}: {str(node)}\n"
        return string

In [19]:
# Loading the datasets

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]
    
with open("data/iris/iris.data", "r") as f:
    iris_data = f.readlines()
iris_ds = []
for line in iris_data[:-1]:
    line = line.strip().split(',')
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    iris_ds.append((x, y))

with open("data/ecoli/ecoli.data", "r") as f:
    ecoli_data = f.readlines()
ecoli_ds = []
for line in ecoli_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    ecoli_ds.append((x, y))

with open("data/image_seg/segmentation.data", "r") as f:
    seg_train_data = f.readlines()
with open("data/image_seg/segmentation.test", "r") as f:
    seg_test_data = f.readlines()
seg_train = []
for line in seg_train_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_train.append((x, y))
seg_test = []
for line in seg_test_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_test.append((x, y))

with open("data/waveform/waveform.data", "r") as f:
    waveform_data = f.readlines()
wave_ds = []
for line in waveform_data:
    line = line.strip().split(",")
    y = line[-1]
    x = np.array(line[:-1], dtype=float)
    wave_ds.append((x, y))

with open("data/yeast/yeast.data", "r") as f:
    yeast_data = f.readlines()
yeast_ds = []
for line in yeast_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    yeast_ds.append((x, y))

with open("data/anuran/Frogs_MFCCs.csv", "r") as f:
    anuran_data = f.readlines()
anuran_ds = []
for line in anuran_data[1:]:
    line = line.strip().split(",")
    x = np.array(line[:-4], dtype=float)
    y = tuple(line[-4:-1])
    anuran_ds.append((x, y))

with open("data/spambase/spambase.data", "r") as f:
    spam_data = f.readlines()
spam_ds = []
for line in spam_data:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    spam_ds.append((x, y))

with open("data/letter/letter-recognition.data", "r") as f:
    letter_data = f.readlines()
letter_ds = []
for line in letter_data:
    line = line.strip().split(",")
    x = np.array(line[1:], dtype=float)
    y = line[0]
    letter_ds.append((x, y))

In [ ]:
def initial_width(ds, delta=1):
    avg_dist = pdist([x for x,_ in ds], metric="euclidean").mean()
    return avg_dist

iris_A = initial_width(iris_ds)
anuran_A = initial_width(anuran_ds)
spam_A = initial_width(spam_ds)
seg_A = initial_width(seg_train)
wave_A = initial_width(wave_ds)
letter_A = initial_width(letter_ds)
yeast_A = initial_width(yeast_ds)
# The implementation of confidence based VEBF takes array of width as initial width
iris_A = np.array([iris_A] * iris_dim)
anuran_A = np.array([anuran_A] * anuran_dim)
spam_A = np.array([spam_A] * spam_dim)
seg_A = np.array([seg_A] * seg_dim)
wave_A = np.array([wave_A] * wave_dim)
letter_A = np.array([letter_A] * letter_dim)
yeast_A = np.array([yeast_A] * yeast_dim)


iris_dim = len(iris_ds[0][0])
anuran_dim = len(anuran_ds[0][0])
spam_dim = len(spam_ds[0][0])
seg_dim = len(seg_train[0][0])
wave_dim = len(wave_ds[0][0])
letter_dim = len(letter_ds[0][0])
yeast_dim = len(yeast_ds[0][0])


In [14]:
def plot_superellipsoid_2d(
    C: np.ndarray,
    U: np.ndarray,
    A: np.ndarray,
    P: np.ndarray,
    ax=None,
    resolution: int = 500,
    fill: bool = False,
    fill_alpha: float = 0.25,
    color: str = "steelblue",
    label: str = None,
):
    """
    Visualize a 2-D superellipsoid defined by the implicit function:

        f(x, y) = |u1 · (r - C) / a1|^p1 + |u2 · (r - C) / a2|^p2  ≤  1

    where u1, u2 are the columns of U (axis directions).

    Parameters
    ----------
    C : ndarray, shape (2,)
        Center of the superellipsoid.
    U : ndarray, shape (2, 2)
        Each *column* is a unit vector giving the direction of an axis.
        (Follows the convention where U[:, i] is the i-th axis direction.)
    A : ndarray, shape (2,)
        Half-lengths along each axis.
    P : ndarray, shape (2,)
        Exponents [p1, p2].  p=2 gives an ellipse; p→∞ gives a rectangle;
        0 < p < 2 gives a "pinched" / star shape.
    ax : matplotlib Axes, optional
        Target axes.  A new figure is created when None.
    resolution : int
        Number of grid points per axis for the implicit-function evaluation.
    fill : bool
        Whether to flood-fill the interior.
    fill_alpha : float
        Alpha for the filled region.
    color : str
        Colour used for both the boundary and (if fill=True) the interior.
    label : str, optional
        Legend label.

    Returns
    -------
    ax : matplotlib Axes
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    # ------------------------------------------------------------------
    # 1.  Build a world-space bounding box that safely encloses the shape
    # ------------------------------------------------------------------
    # The extreme world-space extent along each world axis is bounded by
    # the sum of |Uij| * Aj over the body axes j.
    half_extents = np.abs(U) @ A          # shape (2,)
    margin = 0.15 * half_extents.max()

    x_min, x_max = C[0] - half_extents[0] - margin, C[0] + half_extents[0] + margin
    y_min, y_max = C[1] - half_extents[1] - margin, C[1] + half_extents[1] + margin

    xs = np.linspace(x_min, x_max, resolution)
    ys = np.linspace(y_min, y_max, resolution)
    XX, YY = np.meshgrid(xs, ys)

    # ------------------------------------------------------------------
    # 2.  Evaluate the implicit function in the body frame
    # ------------------------------------------------------------------
    # Translate to the centre
    dX = XX - C[0]          # (res, res)
    dY = YY - C[1]

    # Project onto each body axis:  local_i = U[:, i] · (r - C)
    #   U has shape (2, 2); U[:, 0] is first axis direction, etc.
    local = np.stack(
        [U[0, i] * dX + U[1, i] * dY for i in range(2)],
        axis=0,
    )                        # (2, res, res)

    # Normalise by semi-axes and raise to the respective powers
    F = sum(
        (np.abs(local[i]) / A[i]) ** P[i]
        for i in range(2)
    )                        # (res, res)

    # ------------------------------------------------------------------
    # 3.  Draw: filled contour at F=1 (interior) + boundary contour
    # ------------------------------------------------------------------
    if fill:
        ax.contourf(XX, YY, F, levels=[0, 1], colors=[color], alpha=fill_alpha)

    cs = ax.contour(XX, YY, F, levels=[1.0], colors=[color], linewidths=2)

    # Attach a legend proxy if requested
    if label is not None:
        proxy = mpatches.Patch(facecolor=color, alpha=fill_alpha,
                               edgecolor=color, label=label)
        ax.add_patch(plt.Rectangle((0, 0), 0, 0, visible=False))  # dummy
        handles, labels = ax.get_legend_handles_labels()
        handles.append(proxy)
        labels.append(label)
        ax.legend(handles, labels)

    #ax.set_aspect("equal")
    #ax.grid(True, linestyle="--", alpha=0.4)
    return ax

In [59]:
anuran_result = []
for i in tqdm(range(30)):

    # shuffle the train/test split
    anuran_train, anuran_test = train_test_split(anuran_ds)
    model = VEBF(dimension=anuran_dim, merge_parameter=0., default_width=anuran_A)
    for idx in range(len(anuran_train)):
        x,y = anuran_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    # test the model
    correct = 0
    for x, label in anuran_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    anuran_result.append(correct/len(anuran_test))

100%|██████████| 30/30 [00:59<00:00,  1.98s/it]


In [60]:
spam_result = []
for i in tqdm(range(30)):

    # shuffle the train/test split
    spam_train, spam_test = train_test_split(spam_ds)
    model = VEBF(dimension=spam_dim, merge_parameter=0., default_width=spam_A)
    for idx in range(len(spam_train)):
        x,y = spam_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    # test the model
    correct = 0
    for x, label in spam_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    spam_result.append(correct/len(spam_test))

 17%|█▋        | 5/30 [00:13<01:06,  2.66s/it]/tmp/ipykernel_5154/3384164292.py:51: RuntimeWarning: overflow encountered in power
  return np.sum(np.abs(xs_local / As) ** p, axis=1) - (k ** 2)              # (n,)
100%|██████████| 30/30 [01:21<00:00,  2.71s/it]


In [61]:
letter_result = []
for i in tqdm(range(30)):

    # shuffle the train/test split
    letter_train, letter_test = train_test_split(letter_ds)
    model = VEBF(dimension=letter_dim, merge_parameter=0., default_width=letter_A)
    for idx in range(len(letter_train)):
        x,y = letter_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    # test the model
    correct = 0
    for x, label in letter_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    letter_result.append(correct/len(letter_test))

  0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_5154/3384164292.py:51: RuntimeWarning: overflow encountered in power
  return np.sum(np.abs(xs_local / As) ** p, axis=1) - (k ** 2)              # (n,)
 27%|██▋       | 8/30 [00:28<01:18,  3.56s/it]/tmp/ipykernel_5154/3384164292.py:28: RuntimeWarning: overflow encountered in power
  return np.sum(np.abs(x_local / A) ** p) - (k ** 2)
100%|██████████| 30/30 [01:48<00:00,  3.63s/it]


In [62]:
seg_result = []
for i in tqdm(range(30)):

    # shuffle the train/test split
    random.shuffle(seg_train)
    model = VEBF(dimension=seg_dim, merge_parameter=0., default_width=seg_A)
    for idx in range(len(seg_train)):
        x,y = seg_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    # test the model
    correct = 0
    for x, label in seg_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    seg_result.append(correct/len(seg_test))

100%|██████████| 30/30 [00:05<00:00,  5.48it/s]


In [30]:
def print_stat(arr, name):
    print(f"{name} Mean:", np.mean(arr))
    print(f"{name} Max:", np.max(arr))
    print(f"{name} Min:", np.min(arr))

In [31]:
# 2
print_stat(anuran_result, "anuran")

anuran Mean: 0.9160296502200604
anuran Max: 0.9423210562890897
anuran Min: 0.8839471855455178


In [63]:
print_stat(anuran_result, "anuran")
print()
print_stat(spam_result, "spam")
print()
print_stat(seg_result, "seg")
print()
print_stat(letter_result, "letter")
print()

anuran Mean: 0.9170257123002085
anuran Max: 0.9562195969423211
anuran Min: 0.8665740097289785

spam Mean: 0.8226565327542529
spam Max: 0.8653637350705755
spam Min: 0.50271444082519

seg Mean: 0.5930476190476189
seg Max: 0.6866666666666666
seg Min: 0.4338095238095238

letter Mean: 0.625825
letter Max: 0.672
letter Min: 0.5795

